In [2]:
import boto3
import os
from pathlib import Path
from dotenv import load_dotenv
import sys

env_path = Path.cwd().parent.parent.joinpath("env").joinpath("dev.aws.env")

if env_path.exists:
    load_dotenv(dotenv_path=env_path)
    print("env loaded successfully. App Name = ", os.getenv("APP_NAME"))
else:
    print("Env file not found")

aws_url = os.getenv("AWS_URL")
region = os.getenv("REGION")
aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY")

env loaded successfully. App Name =  AWS PRACTICE


In [2]:
snsClient = boto3.client('sns',
    endpoint_url= aws_url,
    region_name=region,
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key
)

snsClient

In [18]:
sqsClient = boto3.client('sqs',
    endpoint_url= aws_url,
    region_name=region,
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key
 )


In [20]:
# create new queue
response = sqsClient.create_queue(
    QueueName='MyLocalTestingQueue',
    Attributes={
        'DelaySeconds': '0',
        'MessageRetentionPeriod': '86400' # Retain for 1 day (in seconds)
    }
)

print(f"Queue Created successfully! URL: {response['QueueUrl']}")

Queue Created successfully! URL: http://localhost:4566/000000000000/MyLocalTestingQueue


In [170]:
# Receive message from queue
queue_url = "http://localhost:4566/000000000000/MyLocalTestingQueue"
message_from_queue =  sqsClient.receive_message(
    QueueUrl=queue_url
)

if message_from_queue.get("Messages") is  None:
    print("No messages received from queue or queue processed")
else:
    for msg in message_from_queue.get("Messages"):
        print(msg.get("Body"))

        # sqsClient.delete_message(
        #     QueueUrl=queue_url,
        #     ReceiptHandle=msg['ReceiptHandle']
        # )


No messages received from queue or queue processed


In [98]:
snsClient.create_topic(Name="Test topic for SNS")

{'TopicArn': 'arn:aws:sns:us-east-1:000000000000:Test topic for SNS',
 'ResponseMetadata': {'RequestId': '29393f4a-71dc-4b6b-ab2b-add0cda45f0b',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'content-type': 'application/xml',
   'access-control-allow-origin': '*',
   'access-control-allow-methods': 'GET, POST, PUT, DELETE, HEAD, OPTIONS, PATCH',
   'access-control-allow-headers': '*',
   'access-control-expose-headers': '*',
   'x-amzn-requestid': '876caeac-2528-4d6b-9684-6b6dbd43c338',
   'x-amz-request-id': '876caeac-2528-4d6b-9684-6b6dbd43c338',
   'x-amz-id-2': 'Y7qhJVOATkVeRXfZ2BBfSNEhoxFx33BkTmX9w3z7E0pImoXo/3IKWiHAtFVrStYO',
   'content-length': '339',
   'date': 'Mon, 29 Jun 2026 10:51:25 GMT',
   'server': 'hypercorn-h11'},
  'RetryAttempts': 0}}

In [5]:
response = snsClient.create_topic(
    Name="my-strict-ordering-topic.fifo",
    Attributes={
        'FifoTopic': 'true',
        'ContentBasedDeduplication': 'true' # Optional auto-deduplication
    }
)

response

{'TopicArn': 'arn:aws:sns:us-east-1:000000000000:my-strict-ordering-topic.fifo',
 'ResponseMetadata': {'RequestId': 'a3a43ab2-47b7-4b52-a166-6d5f7a674456',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'content-type': 'application/xml',
   'access-control-allow-origin': '*',
   'access-control-allow-methods': 'GET, POST, PUT, DELETE, HEAD, OPTIONS, PATCH',
   'access-control-allow-headers': '*',
   'access-control-expose-headers': '*',
   'x-amzn-requestid': 'cac2cc7d-9dbc-49f8-9131-26dec5af5730',
   'x-amz-request-id': 'cac2cc7d-9dbc-49f8-9131-26dec5af5730',
   'x-amz-id-2': 'mWkwZ0I52Fbuz6Y++tpdhEJF3XYICv3keXOb5Nlj1O9zKxq5kBWIyndN0tsWBqIG',
   'content-length': '350',
   'date': 'Mon, 29 Jun 2026 09:07:49 GMT',
   'server': 'hypercorn-h11'},
  'RetryAttempts': 0}}

In [99]:
# Subscribe an email address to the topic
subscription = snsClient.subscribe(
    TopicArn="arn:aws:sns:us-east-1:000000000000:Test topic for SNS",
    Protocol='sqs',
    Endpoint='arn:aws:sqs:us-east-1:000000000000:MyLocalTestingQueue'
)

print(f"Subscription ARN: {subscription['SubscriptionArn']}")
print("Note: Check your inbox to confirm the pending subscription confirmation.")


Subscription ARN: arn:aws:sns:us-east-1:000000000000:Test topic for SNS:2fbfb333-4641-413d-8c6f-01eacaaaa14a
Note: Check your inbox to confirm the pending subscription confirmation.


In [162]:
#publish new item to topic
publish_response = snsClient.publish(
    TopicArn="arn:aws:sns:us-east-1:000000000000:Test topic for SNS",
    Message='{"order_id": "12341", "status": "shipped"}',
    Subject='Order Status Update'
)

In [3]:
#read secret manager data
sm_client = boto3.client("secretsmanager",
    endpoint_url= aws_url,
    region_name=region,
    aws_access_key_id=aws_access_key_id,
                         aws_secret_access_key=aws_secret_access_key
 )